# Phase 6: Second architecture and held out corruptions

**GPU T4 x2. Internet OFF. Attach the Phase 1 notebook output.**

Nothing is downloaded. CIFAR-10, CIFAR-10-C and the three ResNet18 checkpoints
all come from the Phase 1 output.

Produces `logits2.npz` with:
- WideResNet-16-4, 3 seeds, val + test + all 19 corruptions x 5 severities
- ResNet18 (existing checkpoints, no retraining) on the 4 held out corruptions

Run with Save Version > Save & Run All. Roughly 90 minutes.

In [4]:
import os, glob, time, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision as tv, torchvision.transforms as T

IN   = os.path.dirname(glob.glob("/kaggle/input/**/logits.npz", recursive=True)[0])
CDIR = f"{IN}/CIFAR-10-C"; DATA = f"{IN}/data"
WORK = "/kaggle/working"; DEV = "cuda"
MEAN, STD = (0.4914,0.4822,0.4465), (0.2470,0.2435,0.2616)
SEEDS=[0,1,2]; EPOCHS=60; BATCH=128; VAL_N=5000

STD15=["gaussian_noise","shot_noise","impulse_noise","defocus_blur","glass_blur",
       "motion_blur","zoom_blur","snow","frost","fog","brightness","contrast",
       "elastic_transform","pixelate","jpeg_compression"]
EXTRA=["speckle_noise","gaussian_blur","spatter","saturate"]
ALL19=STD15+EXTRA
print(IN); print(torch.cuda.get_device_name(0))
assert all(os.path.exists(f"{CDIR}/{c}.npy") for c in ALL19), "missing corruption files"

/kaggle/input/notebooks/shayamahmad74/phase-1-sep-14
Tesla T4


## 1. Data, identical split to Phase 1

In [5]:
tf_tr=T.Compose([T.RandomCrop(32,padding=4),T.RandomHorizontalFlip(),T.ToTensor(),T.Normalize(MEAN,STD)])
tf_ev=T.Compose([T.ToTensor(),T.Normalize(MEAN,STD)])
full =tv.datasets.CIFAR10(DATA,train=True ,download=False,transform=tf_tr)
fullv=tv.datasets.CIFAR10(DATA,train=True ,download=False,transform=tf_ev)
test =tv.datasets.CIFAR10(DATA,train=False,download=False,transform=tf_ev)

rng=np.random.default_rng(12345)                    # SAME seed as Phase 1
perm=rng.permutation(len(full)); val_idx,tr_idx=perm[:VAL_N],perm[VAL_N:]
dl=lambda d,s: DataLoader(d,batch_size=BATCH if s else 512,shuffle=s,
                          num_workers=2,pin_memory=True,drop_last=s)
tr_dl =dl(Subset(full ,tr_idx),True)
val_dl=dl(Subset(fullv,val_idx),False)
test_dl=dl(test,False)

old=np.load(f"{IN}/logits.npz")
assert (np.array([fullv[i][1] for i in val_idx])==old["y_val"]).all(), "split mismatch"
print("split verified against Phase 1")

split verified against Phase 1


## 2. WideResNet-16-4

In [6]:
class Block(nn.Module):
    def __init__(s,i,o,stride):
        super().__init__()
        s.bn1=nn.BatchNorm2d(i); s.c1=nn.Conv2d(i,o,3,stride,1,bias=False)
        s.bn2=nn.BatchNorm2d(o); s.c2=nn.Conv2d(o,o,3,1,1,bias=False)
        s.sc=nn.Conv2d(i,o,1,stride,bias=False) if (i!=o or stride!=1) else None
    def forward(s,x):
        h=F.relu(s.bn1(x)); z=s.c1(h); z=s.c2(F.relu(s.bn2(z)))
        return z+(s.sc(h) if s.sc is not None else x)

class WRN(nn.Module):
    def __init__(s,depth=16,k=4,nc=10):
        super().__init__(); n=(depth-4)//6; w=[16,16*k,32*k,64*k]
        s.conv=nn.Conv2d(3,w[0],3,1,1,bias=False); layers=[]
        for i in range(3):
            for j in range(n):
                layers.append(Block(w[i] if j==0 else w[i+1], w[i+1], (1 if i==0 else 2) if j==0 else 1))
        s.blocks=nn.Sequential(*layers); s.bn=nn.BatchNorm2d(w[3]); s.fc=nn.Linear(w[3],nc)
    def forward(s,x):
        z=s.blocks(s.conv(x)); z=F.relu(s.bn(z))
        return s.fc(F.adaptive_avg_pool2d(z,1).flatten(1))

def resnet18_cifar():
    m=tv.models.resnet18(weights=None,num_classes=10)
    m.conv1=nn.Conv2d(3,64,3,1,1,bias=False); m.maxpool=nn.Identity(); return m
print(sum(p.numel() for p in WRN().parameters())/1e6,"M params")

2.74889 M params


## 3. Train WRN

In [7]:
def train(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    m=WRN().to(DEV).to(memory_format=torch.channels_last)
    opt=torch.optim.SGD(m.parameters(),lr=0.1,momentum=0.9,weight_decay=5e-4,nesterov=True)
    sch=torch.optim.lr_scheduler.OneCycleLR(opt,0.1,epochs=EPOCHS,steps_per_epoch=len(tr_dl))
    sc=torch.cuda.amp.GradScaler()
    for ep in range(EPOCHS):
        m.train()
        for x,y in tr_dl:
            x=x.to(DEV,non_blocking=True).to(memory_format=torch.channels_last)
            y=y.to(DEV,non_blocking=True); opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(): loss=F.cross_entropy(m(x),y)
            sc.scale(loss).backward(); sc.step(opt); sc.update(); sch.step()
        if (ep+1)%20==0: print(f"  seed {seed} ep {ep+1} loss {loss.item():.3f}",flush=True)
    return m

wrn={}
for s in SEEDS:
    p=f"{WORK}/wrn_{s}.pt"
    m=WRN().to(DEV)
    if os.path.exists(p): m.load_state_dict(torch.load(p))
    else:
        t0=time.time(); m=train(s); torch.save(m.state_dict(),p)
        print(f"seed {s}: {(time.time()-t0)/60:.1f} min")
    wrn[s]=m.eval()

res={}
for s in SEEDS:
    m=resnet18_cifar().to(DEV); m.load_state_dict(torch.load(f"{IN}/ckpt_{s}.pt")); res[s]=m.eval()
print("models ready")

/tmp/ipykernel_58/959194355.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  sc=torch.cuda.amp.GradScaler()
/tmp/ipykernel_58/959194355.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(): loss=F.cross_entropy(m(x),y)


  seed 0 ep 20 loss 0.262
  seed 0 ep 40 loss 0.193
  seed 0 ep 60 loss 0.004
seed 0: 10.5 min
  seed 1 ep 20 loss 0.370
  seed 1 ep 40 loss 0.199
  seed 1 ep 60 loss 0.005
seed 1: 10.6 min
  seed 2 ep 20 loss 0.380
  seed 2 ep 40 loss 0.189
  seed 2 ep 60 loss 0.014
seed 2: 10.6 min
models ready


## 4. Logit dump

In [8]:
MU=torch.tensor(MEAN).view(1,3,1,1); SD=torch.tensor(STD).view(1,3,1,1)
@torch.no_grad()
def from_arr(m,x):
    o=[]
    for i in range(0,len(x),1000):
        b=torch.from_numpy(x[i:i+1000]).permute(0,3,1,2).float().div_(255)
        with torch.cuda.amp.autocast(): o.append(m(((b-MU)/SD).to(DEV)).float().cpu())
    return torch.cat(o).half().numpy()
@torch.no_grad()
def from_dl(m,dl_): return torch.cat([m(x.to(DEV)).float().cpu() for x,_ in dl_]).half().numpy()

store={"y_val":old["y_val"],"y_test":old["y_test"],"y_corr":old["y_corr"]}
for s in SEEDS:
    store[f"wrn_val_{s}"]=from_dl(wrn[s],val_dl); store[f"wrn_test_{s}"]=from_dl(wrn[s],test_dl)

for c_ in ALL19:
    raw=np.load(f"{CDIR}/{c_}.npy")
    for sev in range(1,6):
        x=raw[(sev-1)*10000:sev*10000]
        for s in SEEDS:
            store[f"wrn_{c_}_{sev}_{s}"]=from_arr(wrn[s],x)
            if c_ in EXTRA: store[f"res_{c_}_{sev}_{s}"]=from_arr(res[s],x)
    print(c_,flush=True)

np.savez_compressed(f"{WORK}/logits2.npz",**store)
print(f"{os.path.getsize(WORK+'/logits2.npz')/1e6:.1f} MB")

/tmp/ipykernel_58/39707096.py:7: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(): o.append(m(((b-MU)/SD).to(DEV)).float().cpu())


gaussian_noise
shot_noise
impulse_noise
defocus_blur
glass_blur
motion_blur
zoom_blur
snow
frost
fog
brightness
contrast
elastic_transform
pixelate
jpeg_compression
speckle_noise
gaussian_blur
spatter
saturate
63.8 MB


## 5. Sanity check

In [9]:
def ac(l,y):
    import numpy as np
    p=torch.softmax(torch.from_numpy(l).float(),1); cf,pr=p.max(1)
    return (pr.numpy()==y).mean(), cf.mean().item()
y=store["y_corr"]
print(f"{'sev':>3} {'WRN acc':>8} {'WRN conf':>9}")
print(f"{0:>3} {ac(store['wrn_test_0'],store['y_test'])[0]*100:7.1f}% "
      f"{ac(store['wrn_test_0'],store['y_test'])[1]*100:8.1f}%")
for sev in range(1,6):
    a,c2=zip(*[ac(store[f"wrn_{k}_{sev}_0"],y) for k in STD15])
    print(f"{sev:>3} {np.mean(a)*100:7.1f}% {np.mean(c2)*100:8.1f}%")

sev  WRN acc  WRN conf
  0    94.0%     96.8%
  1    84.4%     92.8%
  2    77.9%     90.3%
  3    71.6%     87.6%
  4    64.6%     84.8%
  5    53.3%     80.7%


## Next

Re-run Phases 3 to 5 with `logits2.npz`, pointing the loaders at the `wrn_`
prefix. If the coverage error pattern reproduces on a different architecture and
on four corruption types the models never informed, the result is architecture
independent and not an artifact of the 15 standard corruptions.